In [ ]:
from sklearn.model_selection import cross_val_score
# ============================================================
# GA-XGBoost for SEP Event Prediction
# Based on: Research on the occurrence prediction of SEP events
#         based on GA-XGBoost model
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, roc_curve,
    mean_squared_error, mean_absolute_error, r2_score
)
from sklearn.svm import SVC
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
import xgboost as xgb
import shap

# ============================================================
# 1. Data loading and preprocessing
# ============================================================

# Load the dataset
df = pd.read_csv(r"D:\xiangmu\pythonProject\newXGBoost\GAcleaned_2_4_&SEP&paper_total_data.csv", encoding='gbk',
                 index_col=0).reset_index(drop=True)
# df = pd.read_csv("GAcleaned_2_4_&SEP&paper_total_data.csv", sep=',')
df = df.dropna(axis=1, how='all')

# 2. Standardize column names (strip whitespace and convert to uppercase)
df.columns = df.columns.str.strip().str.upper()

# 3. Restore DATE as a column if it is in the index
if 'DATE' not in df.columns:
    df = df.reset_index()

# 4. Inspect the column names again
print("Column names:", df.columns.tolist())

# 5. Verify that all required columns are present
target_columns = ['DATE', 'TIME', 'DT', 'BURST TYPE', 'EVENT TYPE', 'END FREQ']
missing = [c for c in target_columns if c not in df.columns]
if missing:
    print("Warning: the following columns are missing:", missing)
    # Adjust target_columns according to the actual dataset

print("Original dataset shape:", df.shape)
print(df.dtypes)

# Separate features and target labels
target_columns = ['DATE', 'TIME', 'DT', 'BURST TYPE', 'EVENT TYPE', 'End Freq']
X = df.drop(columns=target_columns)
y = df['EVENT TYPE']

# Convert all features to numeric values
X = X.apply(lambda x: pd.to_numeric(x, errors='coerce') if x.dtype == 'object' else x)

# Impute missing values using a rolling mean, followed by the median
X = X.apply(lambda x: x.fillna(x.rolling(window=5, min_periods=1).mean()))
X = X.apply(lambda x: x.fillna(x.median()))

print("Number of features:", X.shape[1])
print("Number of samples:", X.shape[0])
print("Feature names:", list(X.columns))
print("Class distribution:\n", y.value_counts())

# ============================================================
# 2. IQR-based outlier removal and min-max normalization
# ============================================================

def iqr_outlier_removal(X, y, factor=1.5):
    """Remove outliers using the interquartile range (IQR)."""
    mask = pd.Series(True, index=X.index)
    for col in X.columns:
        Q1 = X[col].quantile(0.25)
        Q3 = X[col].quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - factor * IQR
        upper = Q3 + factor * IQR
        mask &= (X[col] >= lower) & (X[col] <= upper)
    X_clean = X[mask].reset_index(drop=True)
    y_clean = y[mask].reset_index(drop=True)
    print(f"Samples remaining after IQR filtering: {len(X_clean)} ({len(X) - len(X_clean)} removed)")
    return X_clean, y_clean

X_clean, y_clean = iqr_outlier_removal(X, y)

# Min-max normalization
scaler = MinMaxScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X_clean), columns=X_clean.columns)
print("Normalization complete; range: [{:.2f}, {:.2f}]".format(X_scaled.min().min(), X_scaled.max().max()))

# ============================================================
# 3. Stratified split by solar cycle and class
# ============================================================

# Infer solar-cycle labels from the DATE year
# Cycle 23: 1996-2008, Cycle 24: 2008-2019, Cycle 25: 2019-2025
if 'CYCLE' in df.columns:
    cycle_labels = df.loc[X_clean.index, 'CYCLE'].reset_index(drop=True)
else:
    cycle_labels = pd.Series(['C1'] * len(X_clean))

# Construct joint stratification labels from cycle and class
stratify_label = cycle_labels.astype(str) + "_" + y_clean.astype(str)
print("Joint stratification label distribution:\n", stratify_label.value_counts())

# Split using joint stratification labels
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_clean,
    test_size=0.3,
    random_state=42,
    stratify=stratify_label
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")
print(f"Training-set class distribution:\n{y_train.value_counts()}")
print(f"Test-set class distribution:\n{y_test.value_counts()}")

# Class weight
neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
scale_pos_weight = neg_count / pos_count
print(f"scale_pos_weight = {scale_pos_weight:.4f}")

# ============================================================
# 4. Evaluation functions
# ============================================================

def evaluate_model(y_true, y_pred, y_prob):
    """Compute all evaluation metrics."""
    cm = confusion_matrix(y_true, y_pred)
    TN, FP, FN, TP = cm.ravel()
    
    acc = accuracy_score(y_true, y_pred)
    pre = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    auc = roc_auc_score(y_true, y_prob)
    
    far = FP / (FP + TP) if (FP + TP) > 0 else 0.0
    pod = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    pofd = FP / (FP + TN) if (FP + TN) > 0 else 0.0
    tss = pod - pofd
    hss_num = 2 * (TP * TN - FP * FN)
    hss_den = (TP + FN) * (FN + TN) + (TP + FP) * (FP + TN)
    hss = hss_num / hss_den if hss_den > 0 else 0.0
    
    mse = mean_squared_error(y_true, y_prob)
    mae = mean_absolute_error(y_true, y_prob)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_prob)
    
    return {
        'Accuracy': acc, 'Precision': pre, 'Recall': rec, 'F1': f1,
        'AUC': auc, 'FAR': far, 'POD': pod, 'TSS': tss, 'HSS': hss,
        'MSE': mse, 'MAE': mae, 'RMSE': rmse, 'R2': r2,
        'TP': TP, 'FP': FP, 'FN': FN, 'TN': TN
    }

def print_metrics(name, metrics):
    print(f"\n===== {name} =====")
    for k, v in metrics.items():
        if k in ['TP', 'FP', 'FN', 'TN']:
            print(f"{k:12s}: {v}")
        else:
            print(f"{k:12s}: {v:.4f}")

# ============================================================
# 5. Baseline XGBoost model (grid search with five-fold cross-validation)
# ============================================================

print("\n" + "="*60)
print("5. Baseline XGBoost model (grid search)")
print("="*60)

param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 4, 5],
    'learning_rate': [0.02, 0.05, 0.1],
    'subsample': [0.7, 0.8, 1.0],
    'colsample_bytree': [0.7, 0.88, 1.0],
    'alpha': [0.05, 0.08, 0.3],
    'lambda': [0.01, 0.2, 0.5]
}

xgb_base = xgb.XGBClassifier(
    random_state=42,
    eval_metric='logloss',
    use_label_encoder=False,
    scale_pos_weight=scale_pos_weight
)

# Five-fold cross-validation within the grid search
grid_search = GridSearchCV(
    estimator=xgb_base,
    param_grid=param_grid,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)
best_xgb = grid_search.best_estimator_

print("\nBest hyperparameters:", grid_search.best_params_)

# Evaluate the baseline model
y_train_pred = best_xgb.predict(X_train)
y_test_pred = best_xgb.predict(X_test)
y_train_prob = best_xgb.predict_proba(X_train)[:, 1]
y_test_prob = best_xgb.predict_proba(X_test)[:, 1]

train_metrics = evaluate_model(y_train, y_train_pred, y_train_prob)
test_metrics = evaluate_model(y_test, y_test_pred, y_test_prob)

print_metrics("Baseline XGBoost - training set", train_metrics)
print_metrics("Baseline XGBoost - test set", test_metrics)

# ============================================================
# 6. Learning curve
# ============================================================

from sklearn.model_selection import learning_curve

train_sizes, train_scores, test_scores = learning_curve(
    best_xgb, X_train, y_train, cv=10,
    train_sizes=np.linspace(0.1, 1, 10),
    scoring='accuracy', n_jobs=-1
)

train_mean = np.mean(train_scores, axis=1)
train_std = np.std(train_scores, axis=1)
test_mean = np.mean(test_scores, axis=1)
test_std = np.std(test_scores, axis=1)

plt.figure(figsize=(8, 5), dpi=150)
plt.plot(train_sizes, train_mean, 'o-', color='#5f9ed1', label='Training Score')
plt.plot(train_sizes, test_mean, 'o-', color='#f9736b', label='Validation Score')
plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.15, color='#5f9ed1')
plt.fill_between(train_sizes, test_mean - test_std, test_mean + test_std, alpha=0.15, color='#f9736b')
plt.xlabel('Sample Size')
plt.ylabel('Accuracy')
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig('learning_curve.png', dpi=300)
plt.show()

# ============================================================
# 7. GA-XGBoost model (genetic algorithm-based hyperparameter optimization)
# ============================================================

print("\n" + "="*60)
print("7. GA-XGBoost model (genetic algorithm optimization)")
print("="*60)

# Genetic algorithm settings
POP_SIZE = 20        # Population size
N_GENERATIONS = 40   # Number of generations
CROSSOVER_PROB = 0.8 # Crossover probability
MUTATION_PROB = 0.1  # Mutation probability

# Hyperparameter search ranges
param_ranges = {
    'n_estimators': (100, 400),
    'max_depth': (3, 9),
    'learning_rate': (0.02, 0.2),
    'subsample': (0.6, 1.0),
    'colsample_bytree': (0.6, 1.0),
    'alpha': (0, 1),
    'lambda': (0, 1),
    'gamma': (0, 4)
}
param_names = list(param_ranges.keys())
D = len(param_names)

def decode_individual(x):
    """Decode a real-valued vector into a hyperparameter dictionary."""
    params = {}
    for i, name in enumerate(param_names):
        lo, hi = param_ranges[name]
        val = lo + x[i] * (hi - lo)
        if name in ['n_estimators', 'max_depth']:
            params[name] = int(round(val))
        else:
            params[name] = float(val)
    return params

def fitness(x):
    """Fitness objective: F = -AUC."""
    params = decode_individual(x)
    # Enforce valid hyperparameter values
    params['n_estimators'] = max(10, params['n_estimators'])
    params['max_depth'] = max(1, params['max_depth'])
    
    model = xgb.XGBClassifier(
        **params,
        random_state=42,
        eval_metric='logloss',
        use_label_encoder=False,
        scale_pos_weight=scale_pos_weight
    )
    # AUC from five-fold cross-validation
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)
    return -np.mean(scores)  # Minimize negative AUC

# Initialize the population
np.random.seed(42)
population = np.random.rand(POP_SIZE, D)
fitness_history = []
best_individual = None
best_fitness = float('inf')

# Iterative optimization
for gen in range(N_GENERATIONS):
    # Evaluate fitness
    fitness_values = np.array([fitness(ind) for ind in population])
    
    # Record the best individual
    min_idx = np.argmin(fitness_values)
    if fitness_values[min_idx] < best_fitness:
        best_fitness = fitness_values[min_idx]
        best_individual = population[min_idx].copy()
    
    fitness_history.append(best_fitness)
    
    if (gen + 1) % 5 == 0:
        print(f"Generation {gen+1:3d} | Best Fitness = {best_fitness:.6f} | Best AUC = {-best_fitness:.6f}")
    
    # Selection (roulette-wheel sampling)
    fitness_shifted = fitness_values - fitness_values.min() + 1e-6
    probs = fitness_shifted / fitness_shifted.sum()
    selected_idx = np.random.choice(POP_SIZE, size=POP_SIZE, p=probs)
    selected = population[selected_idx]
    
    # Crossover (single-point crossover of real-valued vectors)
    offspring = selected.copy()
    for i in range(0, POP_SIZE - 1, 2):
        if np.random.rand() < CROSSOVER_PROB:
            r = np.random.randint(1, D)
            offspring[i, r:] = selected[i+1, r:]
            offspring[i+1, r:] = selected[i, r:]
    
    # Mutation (uniform random replacement)
    for i in range(POP_SIZE):
        if np.random.rand() < MUTATION_PROB:
            j = np.random.randint(D)
            offspring[i, j] = np.random.rand()
    
    population = offspring

# Report the best hyperparameters
best_params = decode_individual(best_individual)
print("\nGA-optimized hyperparameters:", best_params)

# ============================================================
# 8. Train and evaluate the final GA-XGBoost model
# ============================================================

best_ga_xgb = xgb.XGBClassifier(
    **best_params,
    random_state=42,
    eval_metric='logloss',
    use_label_encoder=False,
    scale_pos_weight=scale_pos_weight
)
best_ga_xgb.fit(X_train, y_train)

# Evaluation
y_train_pred_ga = best_ga_xgb.predict(X_train)
y_test_pred_ga = best_ga_xgb.predict(X_test)
y_train_prob_ga = best_ga_xgb.predict_proba(X_train)[:, 1]
y_test_prob_ga = best_ga_xgb.predict_proba(X_test)[:, 1]

train_metrics_ga = evaluate_model(y_train, y_train_pred_ga, y_train_prob_ga)
test_metrics_ga = evaluate_model(y_test, y_test_pred_ga, y_test_prob_ga)

print_metrics("GA-XGBoost - training set", train_metrics_ga)
print_metrics("GA-XGBoost - test set", test_metrics_ga)

# Plot the GA convergence curve
plt.figure(figsize=(8, 5), dpi=150)
plt.plot(range(1, N_GENERATIONS+1), [-f for f in fitness_history], 'b-', linewidth=2)
plt.xlabel('Generation')
plt.ylabel('Best AUC')
plt.title('GA Convergence Curve')
plt.tight_layout()
plt.savefig('ga_convergence.png', dpi=300)
plt.show()

# ============================================================
# 9. SHAP-based feature importance analysis
# ============================================================

print("\n" + "="*60)
print("9. SHAP-based feature importance analysis")
print("="*60)

explainer = shap.TreeExplainer(best_ga_xgb)
shap_values = explainer.shap_values(X_test)

# Rank features by importance
feature_importance = np.abs(shap_values).mean(axis=0)
importance_df = pd.DataFrame({
    'Feature': X_test.columns,
    'Importance': feature_importance
}).sort_values('Importance', ascending=False)

print("\nFeature importance ranking:")
print(importance_df.to_string(index=False))

# Plot the SHAP summary bar chart
plt.figure(figsize=(10, 6), dpi=150)
shap.summary_plot(shap_values, X_test, plot_type="bar", show=False)
plt.tight_layout()
plt.savefig('shap_importance.png', dpi=300)
plt.show()

# Plot the SHAP beeswarm chart
plt.figure(figsize=(10, 6), dpi=150)
shap.summary_plot(shap_values, X_test, show=False)
plt.tight_layout()
plt.savefig('shap_beeswarm.png', dpi=300)
plt.show()

# ============================================================
# 10. Feature selection (threshold: 0.05)
# ============================================================

threshold = 0.05 * importance_df['Importance'].sum()
selected_features = importance_df[importance_df['Importance'] > threshold]['Feature'].tolist()
print(f"\nThreshold = {threshold:.4f}")
print(f"Selected features ({len(selected_features)}): {selected_features}")

# Reduced-feature model
X_train_sel = X_train[selected_features]
X_test_sel = X_test[selected_features]

best_ga_xgb_sel = xgb.XGBClassifier(
    **best_params,
    random_state=42,
    eval_metric='logloss',
    use_label_encoder=False,
    scale_pos_weight=scale_pos_weight
)
best_ga_xgb_sel.fit(X_train_sel, y_train)

# Evaluate the reduced-feature model
y_train_pred_sel = best_ga_xgb_sel.predict(X_train_sel)
y_test_pred_sel = best_ga_xgb_sel.predict(X_test_sel)
y_train_prob_sel = best_ga_xgb_sel.predict_proba(X_train_sel)[:, 1]
y_test_prob_sel = best_ga_xgb_sel.predict_proba(X_test_sel)[:, 1]

train_metrics_sel = evaluate_model(y_train, y_train_pred_sel, y_train_prob_sel)
test_metrics_sel = evaluate_model(y_test, y_test_pred_sel, y_test_prob_sel)

print_metrics("GA-XGBoost (7 features) - training set", train_metrics_sel)
print_metrics("GA-XGBoost (7 features) - test set", test_metrics_sel)

# ============================================================
# 11. Comparison models (SVM, XGBoost, AdaBoost)
# ============================================================

print("\n" + "="*60)
print("11. Comparison models")
print("="*60)

best_th = 0.63

models = {
    'SVM': SVC(probability=True, random_state=42),
    'XGBoost': xgb.XGBClassifier(
        random_state=42, max_depth=2, learning_rate=0.08,
        reg_alpha=2, reg_lambda=2, subsample=0.7,
        colsample_bytree=0.7, scale_pos_weight=scale_pos_weight
    ),
    'AdaBoost': AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=1, random_state=42),
        n_estimators=15, learning_rate=0.3, random_state=42
    ),
    'GA-XGBoost': best_ga_xgb
}

results = {}
for name, model in models.items():
    if name != 'GA-XGBoost':
        model.fit(X_train, y_train)
    
    y_prob = model.predict_proba(X_test)[:, 1]
    y_pred = (y_prob > best_th).astype(int)
    results[name] = evaluate_model(y_test, y_pred, y_prob)

# Print the model comparison table
print(f"\n{'Model':<12} {'Acc':<8} {'Precision':<10} {'Recall':<8} {'F1':<8} {'AUC':<8} {'FAR':<8}")
print("-" * 70)
for name, m in results.items():
    print(f"{name:<12} {m['Accuracy']:.4f}   {m['Precision']:.4f}     {m['Recall']:.4f}   {m['F1']:.4f}   {m['AUC']:.4f}   {m['FAR']:.4f}")

# Confusion matrix counts
print(f"\n{'Model':<12} {'TP':<6} {'FP':<6} {'FN':<6} {'TN':<6}")
print("-" * 40)
for name, m in results.items():
    print(f"{name:<12} {m['TP']:<6} {m['FP']:<6} {m['FN']:<6} {m['TN']:<6}")

# ============================================================
# 12. ROC curves
# ============================================================

plt.figure(figsize=(8, 6), dpi=150)
for name, model in models.items():
    if name != 'GA-XGBoost':
        y_prob = model.predict_proba(X_test)[:, 1]
    else:
        y_prob = best_ga_xgb.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc_val = roc_auc_score(y_test, y_prob)
    plt.plot(fpr, tpr, label=f'{name} (AUC = {auc_val:.4f})', linewidth=2)

plt.plot([0, 1], [0, 1], 'k--', linewidth=1)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve Comparison')
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig('roc_curve.png', dpi=300)
plt.show()

# ============================================================
# 13. Permutation feature importance
# ============================================================

from sklearn.inspection import permutation_importance

perm_result = permutation_importance(
    best_ga_xgb, X_test, y_test,
    n_repeats=30, random_state=42, scoring='roc_auc'
)

perm_df = pd.DataFrame({
    'Feature': X_test.columns,
    'Importance': perm_result.importances_mean,
    'Std': perm_result.importances_std
}).sort_values('Importance', ascending=False)

print("\nPermutation feature importance:")
print(perm_df.to_string(index=False))

plt.figure(figsize=(10, 6), dpi=150)
plt.barh(range(len(perm_df)), perm_df['Importance'].values, color='#1f77b4', alpha=0.85)
plt.yticks(range(len(perm_df)), perm_df['Feature'].values)
plt.gca().invert_yaxis()
plt.xlabel('Permutation Importance (AUC decrease)')
plt.tight_layout()
plt.savefig('permutation_importance.png', dpi=300)
plt.show()

print("\n" + "="*60)
print("The complete workflow has finished.")
print("="*60)

In [ ]:
# ============================================================
# 14. Independent validation using SEPValidation
# ============================================================
print("\n" + "="*60)
print("14. Independent validation using SEPValidation")
print("="*60)

import os
from sklearn.metrics import roc_curve

# ---------------------- 14.1 Load the SEPValidation dataset ----------------------
sepval_path = r"D:\xiangmu\pythonProject\newXGBoost\SEPValidation_dataset.xlsx"

if not os.path.exists(sepval_path):
    raise FileNotFoundError(f"File not found: {sepval_path}")

# Read the XLSX file first; install openpyxl if it is missing
try:
    sepval_df = pd.read_excel(sepval_path)
except Exception:
    sepval_csv = sepval_path.replace(".xlsx", ".csv")
    sepval_df = pd.read_csv(sepval_csv, encoding='utf-8-sig')

print("Original SEPValidation dataset shape:", sepval_df.shape)
print("Original SEPValidation column names:", sepval_df.columns.tolist())

# ---------------------- 14.2 Rename columns ----------------------
# Column M corresponds to 4End Freq; the source label 2End Freq was entered in error.
sepval_df.columns = ['Category', 'Date', '2DT', '4DT', 'Class_Label',
                     'Flare_Magnitude', 'LON', 'CLASS_raw', 'CPA', 'W', 'S',
                     '2End_Freq', '4End_Freq', 'F10.7obs', 'EVENT_TYPE']

# Compute CLASS = log10(Flare Magnitude)
sepval_df['CLASS'] = np.log10(pd.to_numeric(sepval_df['Flare_Magnitude'], errors='coerce'))

# Map columns to the feature names used in training
sepval_df['2End Freq'] = sepval_df['2End_Freq']
sepval_df['4End Freq'] = sepval_df['4End_Freq']
sepval_df['F10.7'] = sepval_df['F10.7obs']

# ---------------------- 14.3 Align features with the training dataset ----------------------
train_features = X.columns.tolist()
print("\nTraining features:", train_features)

missing_feats = [f for f in train_features if f not in sepval_df.columns]
print("Missing features:", missing_feats if missing_feats else "None")

X_sepval = sepval_df[train_features].copy()
y_sepval = sepval_df['EVENT_TYPE'].copy()

print(f"\nSEPValidation sample count: {len(X_sepval)}")
print(f"Class distribution:\n{y_sepval.value_counts()}")

# ---------------------- 14.4 Impute missing values using training-set statistics ----------------------
X_sepval = X_sepval.apply(pd.to_numeric, errors='coerce')
X_sepval = X_sepval.fillna(X[train_features].median())

# ---------------------- 14.5 Normalize using training-set minima and maxima ----------------------
X_sepval_norm = (X_sepval - X[train_features].min()) / \
                (X[train_features].max() - X[train_features].min() + 1e-10)

# Add low-amplitude noise as in the training workflow
np.random.seed(42)
X_sepval_noisy = X_sepval_norm + np.random.normal(0, 0.001, X_sepval_norm.shape)

# ---------------------- 14.6 Predict with the existing model (without retraining) ----------------------
y_sepval_prob = best_ga_xgb.predict_proba(X_sepval_noisy)[:, 1]
y_sepval_pred = (y_sepval_prob > best_th).astype(int)

print(f"\nUsing the threshold selected during training: {best_th:.4f}")

# ---------------------- 14.7 Compute evaluation metrics ----------------------
metrics_sepval = evaluate_model(y_sepval, y_sepval_pred, y_sepval_prob)
print_metrics("GA-XGBoost - SEPValidation", metrics_sepval)

# Confusion matrix counts
print("\nConfusion matrix counts:")
print(f"TP={metrics_sepval['TP']}, FP={metrics_sepval['FP']}, "
      f"FN={metrics_sepval['FN']}, TN={metrics_sepval['TN']}")

# ---------------------- 14.8 Confusion matrix ----------------------
cm_sepval = confusion_matrix(y_sepval, y_sepval_pred)
plt.figure(figsize=(6, 5), dpi=300)
sns.heatmap(cm_sepval, annot=True, fmt='d', cmap='Blues',
            xticklabels=['NonSEP', 'SEP'],
            yticklabels=['NonSEP', 'SEP'],
            annot_kws={'fontsize': 14})
plt.xlabel('Predicted', fontsize=14)
plt.ylabel('True', fontsize=14)
plt.title('Confusion Matrix on SEPValidation', fontsize=14)
plt.tight_layout()
plt.savefig('sepval_confusion_matrix.png', dpi=300)
plt.show()


# ---------------------- 14.9 Compare with the training and test sets ----------------------
train_metrics_final = evaluate_model(y_train, y_train_pred_ga, y_train_prob_ga)
test_metrics_final  = evaluate_model(y_test,  y_test_pred_ga,  y_test_prob_ga)

comparison_df = pd.DataFrame({
    'Dataset': ['Training', 'Testing', 'SEPValidation'],
    'Accuracy': [train_metrics_final['Accuracy'], test_metrics_final['Accuracy'], metrics_sepval['Accuracy']],
    'Precision':[train_metrics_final['Precision'],test_metrics_final['Precision'],metrics_sepval['Precision']],
    'Recall':   [train_metrics_final['Recall'],   test_metrics_final['Recall'],   metrics_sepval['Recall']],
    'F1':       [train_metrics_final['F1'],       test_metrics_final['F1'],       metrics_sepval['F1']],
    'AUC':      [train_metrics_final['AUC'],      test_metrics_final['AUC'],      metrics_sepval['AUC']],
    'FAR':      [train_metrics_final['FAR'],      test_metrics_final['FAR'],      metrics_sepval['FAR']],
    'TSS':      [train_metrics_final['TSS'],      test_metrics_final['TSS'],      metrics_sepval['TSS']],
    'HSS':      [train_metrics_final['HSS'],      test_metrics_final['HSS'],      metrics_sepval['HSS']],
})

print("\n" + "="*90)
print("  GA-XGBoost performance on the training, test, and SEPValidation datasets")
print("="*90)
print(comparison_df.to_string(index=False))

# ---------------------- 14.10 Save results for the Zenodo repository ----------------------
comparison_df.to_csv('sepval_comparison_results.csv', index=False)
pd.DataFrame([metrics_sepval]).to_csv('sepval_evaluation_results.csv', index=False)

# Save the optimal parameters, threshold, and feature lists for reproducibility
import json
repro_info = {
    'best_params': best_params,
    'best_threshold': float(best_th),
    'train_features': train_features,
    'selected_features': selected_features if 'selected_features' in dir() else None,
    'scale_pos_weight': float(scale_pos_weight),
    'random_seed': 42
}
with open('reproducibility_info.json', 'w', encoding='utf-8') as f:
    json.dump(repro_info, f, indent=2, ensure_ascii=False, default=str)

print("\n✅ Validation results saved:")
print("   - sepval_comparison_results.csv")
print("   - sepval_evaluation_results.csv")
print("   - reproducibility_info.json")
print("   - sepval_confusion_matrix.png")
print("   - sepval_roc_curve.png")

print("\n" + "="*60)
print("The complete workflow, including independent SEPValidation validation, has finished.")
print("="*60)